# 13 · Persistent vacancies

Which IT postings stay listed for months, and for which roles?

**Method.** A *vacancy key* is the same company, cleaned title and city group. A key first seen in month *m* is **persistent** if it is listed in at least three of the six months from *m* on; only keys first seen by April 2026 are counted, so every key can be watched for six months. This is a **proxy for "listed for a long time"**: boards repost to stay fresh, and a scraper that revisits a board looks the same as a vacancy that stays open. It is not a time-to-fill and it is never compared across sources, only inside CareerBuilder and LinkedIn.

In [1]:
from IPython.display import display

from vnjobs.analysis import core as a
from vnjobs.analysis import persistence as p

a.apply_style()
con = a.connect()
df = a.load_features(con)

In [2]:
table = p.vacancy_table(df)
a.save_table(
    table.groupby(["source", "persistent"]).size().rename("keys").reset_index(), "13-vacancy-keys"
)
print(f"{len(table):,} vacancy keys; {100 * table.persistent.mean():.1f} % persistent overall")
table.groupby("source").persistent.agg(
    keys="size", pct_persistent=lambda s: round(100 * s.mean(), 1)
).sort_values("keys", ascending=False).head(8)

85,412 vacancy keys; 4.2 % persistent overall


,keys,pct_persistent
source,,
jora_vn,33485,2.9
careerbuilder_vn,19755,5.9
linkedin_vn,15319,4.8
careerjet_vn,5581,4.0
monsterasia_vn,3684,4.2
indeed_vn,2824,1.2
careerviet_vn,1170,11.3
smartrecruiters_vn,834,12.8


## 1. By the year a vacancy first appeared

Persistent share inside CareerBuilder and LinkedIn.

In [3]:
years = p.by_first_year(table)
a.save_table(years, "13-persistent-by-first-year")
years

,year,source,keys,pct_persistent
0,2021,CareerBuilder,4003,7.7
1,2022,CareerBuilder,6057,3.6
2,2023,CareerBuilder,4293,6.5
3,2024,CareerBuilder,2534,5.5
4,2025,CareerBuilder,2731,7.4
5,2023,LinkedIn,2376,6.6
6,2024,LinkedIn,1429,1.9
7,2025,LinkedIn,7901,4.4
8,2026,LinkedIn,3612,5.7


## 2. By role

Persistent share per role, inside each source (roles with at least 200 keys).

In [4]:
for label, source in p.SOURCES.items():
    roles = p.persistence_by(table, "role", source)
    a.save_table(roles, f"13-persistent-by-role-{label.lower()}")
    print(label)
    display(roles)

CareerBuilder


,group,keys,pct_persistent
0,ai_ml,272,9.2
11,security,560,7.5
2,data,1819,6.9
13,technical_non_software,3114,6.8
3,devops_cloud,872,6.4
1,backend,1477,6.3
10,qa_test,2792,6.0
9,other,835,5.4
7,it_support_infra,3315,5.3
5,frontend,581,5.2


LinkedIn


,group,keys,pct_persistent
0,ai_ml,1690,8.4
4,embedded_iot,238,7.6
2,data,1476,5.6
3,devops_cloud,1193,5.4
13,software_other,1572,5.2
7,fullstack,1141,4.5
10,other,321,4.4
1,backend,2365,4.3
8,it_support_infra,916,4.0
11,qa_test,1933,3.8


## 3. Employers

Employers with the highest persistent share (at least 40 vacancy keys), all sources. Treat with care: an employer that keeps a posting live looks persistent whether or not the vacancy is open.

In [5]:
companies = p.persistent_companies(table)
a.save_table(companies, "13-persistent-companies")
companies

,company,keys,pct_persistent
0,Ngân Hàng Tmcp Sài Gòn,111,50.5
1,Ngân Hàng Tmcp Đại Chúng,158,31.6
2,Lotte Finance,51,25.5
3,Action Composites Hightech Industries,46,23.9
4,Grab,122,23.0
5,Smartdev,108,22.2
6,Ngân Hàng Tmcp Quân Đội,270,22.2
7,Gameloft,91,22.0
8,Kms Technology,404,22.0
9,Tập Đoàn Omi Group,68,20.6


## Findings

1. **Persistence is rare.** Of 85,412 vacancy keys (same company, title and city) first seen by April 2026, only 4.2 % were listed in three or more of their first six months: CareerBuilder 5.9 %, LinkedIn 4.8 %, Jora 2.9 %. Most postings appear in one or two months.
2. **AI/ML is the most persistent role in both sources.** CareerBuilder 9.2 % (272 keys) and LinkedIn 8.4 % (1,690 keys), against about 4–6 % for most roles. That fits the idea that AI roles are harder to fill, but the proxy cannot show it. Other roles are less consistent: security is 7.5 % on CareerBuilder but 2.7 % on LinkedIn; data is 6.9 % and 5.6 %; full-stack 3.8 % and 4.5 %; ERP is lowest at about 2.8 % in both.
3. **No trend over the years.** By the year a key was first seen: CareerBuilder 7.7 % (2021), 3.6 % (2022), 6.5 % (2023), 5.5 % (2024), 7.4 % (2025); LinkedIn 6.6 % (2023), 1.9 % (2024), 4.4 % (2025), 5.7 % (2026).
4. **Employers with the highest persistent share** (at least 40 vacancy keys): a Sài Gòn bank 50.5 % (111 keys), PVcomBank 31.6 %, Lotte Finance 25.5 %, Grab 23.0 %, MB Bank 22.2 %, KMS Technology 22.0 %, Gameloft 22.0 %. Banks are over-represented, which may mean continuous recruiting or reposting rather than unfilled posts.

**What this notebook cannot say:** how long a job really stays open, or whether it was ever filled. Each posting is captured once, when it appears, so there is no closing date in the feed.
